# 02 — Baseline Random Forest

Sirve como **techo de referencia** antes de pasar a anomalía + clasificador. Entrenamos dos versiones:

1. **Modelo full**: las 84 features de InSDN (menos las que filtran etiqueta).
2. **Modelo deployable**: solo las 7 features derivables desde stats de OpenFlow.

Comparar ambos nos dice cuánto rendimiento sacrificamos al limitarnos a lo que la Ryu app puede medir.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, str(Path.cwd().parent))
from src.data import get_insdn_path
from src.features import (
    clean_insdn,
    INSDN_TO_OPENFLOW,
    INSDN_DROP_COLS,
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 1. Carga y limpieza

In [ ]:
csv = next(get_insdn_path().rglob('*.csv'))
df = pd.read_csv(csv, low_memory=False)
df.columns = df.columns.str.strip()
df = clean_insdn(df)
print(f'Shape tras limpieza: {df.shape}')
df['Label'].value_counts()

## 2. Preparación de los dos conjuntos de features

**Full**: todas las columnas numéricas menos identificadores (`Flow ID`, IPs, `Timestamp`) que provocarían fuga.

**Deployable**: solo las 7 columnas mapeables a OpenFlow.

In [ ]:
label = df['Label']

# Full: numéricas - drop_cols
drop = set(INSDN_DROP_COLS + ['Label'])
X_full = df.drop(columns=[c for c in drop if c in df.columns]).select_dtypes(include=[np.number])

# Deployable: solo las columnas mapeadas
deployable_cols = list(INSDN_TO_OPENFLOW.keys())
X_deploy = df[deployable_cols].copy()
X_deploy['Flow Duration'] = X_deploy['Flow Duration'] / 1e6  # µs -> s
X_deploy = X_deploy.rename(columns=INSDN_TO_OPENFLOW)

print(f'Full:       {X_full.shape[1]} features')
print(f'Deployable: {X_deploy.shape[1]} features -> {list(X_deploy.columns)}')

In [ ]:
le = LabelEncoder()
y = le.fit_transform(label)
class_names = le.classes_
print('Clases:', dict(zip(class_names, np.bincount(y))))

## 3. Entrenamiento

Split estratificado 80/20. Random Forest con 200 árboles (suficiente para baseline, no optimizamos hiperparámetros aquí).

In [ ]:
def train_and_eval(X, y, name):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    clf = RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight='balanced',  # compensar U2R / BFA minoritarias
    )
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)

    print(f'=== {name} ===')
    print(f'F1 macro:    {f1_score(y_te, y_pred, average="macro"):.4f}')
    print(f'F1 weighted: {f1_score(y_te, y_pred, average="weighted"):.4f}')
    try:
        auc = roc_auc_score(y_te, y_proba, multi_class="ovr", average="macro")
        print(f'ROC-AUC (OvR macro): {auc:.4f}')
    except ValueError as e:
        print(f'ROC-AUC: no calculable ({e})')
    print('\n', classification_report(y_te, y_pred, target_names=class_names, digits=4))
    return clf, X_te, y_te, y_pred

In [ ]:
clf_full, Xte_full, yte_full, yp_full = train_and_eval(X_full, y, 'Modelo FULL (84 features)')

In [ ]:
clf_dep, Xte_dep, yte_dep, yp_dep = train_and_eval(X_deploy, y, 'Modelo DEPLOYABLE (7 features OpenFlow)')

## 4. Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (y_te, y_p, name) in zip(
    axes,
    [(yte_full, yp_full, 'Full'), (yte_dep, yp_dep, 'Deployable')],
):
    cm = confusion_matrix(y_te, y_p, normalize='true')
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names, ax=ax,
    )
    ax.set_title(f'{name} — matriz de confusión normalizada')
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout()

## 5. Importancia de features (modelo full)

Sirve para entender qué columnas pesan más en el modelo grande — pista de qué buscar si el deployable rinde mucho peor.

In [ ]:
importances = pd.Series(clf_full.feature_importances_, index=X_full.columns)
top20 = importances.sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
top20.iloc[::-1].plot(kind='barh', ax=ax)
ax.set_title('Top 20 features por importancia (RF full)')
plt.tight_layout()

## Próximos pasos

- Guardar la versión limpia y los splits en `data/processed/` para no repetir trabajo en el resto de notebooks.
- Pasar a `03_anomaly_detector.ipynb`: entrenar Autoencoder / Isolation Forest **solo con tráfico Normal** y definir el umbral dinámico.